In [1]:
import logfire
import os

logfire.configure(
    token=os.environ["LOGFIRE_TOKEN"],
    service_name="ai-assist-demo",
    scrubbing=False,
    send_to_logfire=True,
)

# Getting started

Let's walk through a small but realistic example. The task is **product review sentiment analysis**: given a customer review, predict whether the sentiment is `positive`, `neutral`, or `negative`, and produce a short justification for the call.

We start with a tiny labelled dataset. The `reference` is the ground-truth label.


In [2]:
from lmeh.datatypes import Example

examples = [
    Example(
        inputs={
            "review": "Battery lasts two full days and the screen is gorgeous. Best phone I've owned."
        },
        reference="positive",
    ),
    Example(
        inputs={
            "review": "It works. Setup was fine, nothing surprising, nothing to complain about."
        },
        reference="positive",
    ),
    Example(
        inputs={"review": "Stopped charging after three weeks. Support never replied. Avoid."},
        reference="negative",
    ),
    Example(
        inputs={
            "review": "    Camera   is   AMAZING!!!   colors pop, low-light is great.\n\n\nHighly recommend."
        },
        reference="positive",
    ),
    Example(
        inputs={
            "review": "Arrived on time. Packaging was a bit beaten up but the product itself looks ok."
        },
        reference="neutral",
    ),
]

Now we define the target function — the **thing we actually want to evaluate**. Note that it is not just a thin wrapper around `complete()`: there can be real pre- and post-processing around the model call. That's intentional. In production, what users hit is rarely the raw completion; it's the **completion plus the scaffold around it**. The harness lets us evaluate that full product.

The function must adhere to the `TargetFunction` protocol: take its **named inputs**, the **prompt template**, and an **LM config**, then return whatever the downstream scorers should consume. The harness unpacks `Example.inputs` as keyword arguments, so the parameter names must match the dict keys.

The underlying `CompletionRequest` / `CompletionResponse` are captured automatically (via `lmdk.observe`) and attached to the trial — the target doesn't need to surface them.
    

In [3]:
import re
from typing import Literal

from lmdk import complete, render_template
from pydantic import BaseModel, Field

from lmeh.datatypes import LMConfig

ALLOWED_LABELS = {"positive", "neutral", "negative"}
MAX_REVIEW_CHARS = 500

def classify_sentiment(review: str, prompt_template: str, config: LMConfig) -> dict:
    # Pre-processing: normalise whitespace and cap length
    cleaned = re.sub(r"\s+", " ", review).strip()
    if len(cleaned) > MAX_REVIEW_CHARS:
        cleaned = cleaned[:MAX_REVIEW_CHARS] + "…"

    # Define the output schema expected from the model
    class Output(BaseModel):
        sentiment: Literal["positive", "neutral", "negative"]
        reason: str = Field(description="One short sentence justifying the sentiment label.")

    # Call the model with lmdk.complete
    prompt = render_template(template=prompt_template, REVIEW=cleaned)
    response = complete(
        model=config.model,
        generation_kwargs=config.generation_kwargs,
        prompt=prompt,
        output_schema=Output,
    )

    # Post-processing: normalize the label and fall back to"neutral"
    label = (response.output.sentiment or "").strip().lower()
    if label not in ALLOWED_LABELS:
        label = "neutral"

    # Return arbitrary format that will be downstream consumed
    return {"sentiment": label, "reason": response.output.reason.strip()}

Now, lets define the moving parts under test. The two sweepable axes are the **prompt template** and the **LM config** (model, generation kwargs).


In [4]:
prompt_template = """Your task is to classify the sentiment of this product review:

{{ REVIEW }}

The possible labels are 'positive', 'neutral' and 'negative'
"""

config = LMConfig(
    model="mistral:mistral-small-latest",
    generation_kwargs={"temperature": 0.2},
)

With these ingredients, we can already run a trial: execute the target function on one example with the config under test.


In [5]:
from lmeh.execution import run_trial

trial = run_trial(
    target=classify_sentiment,
    prompt_template=prompt_template,
    config=config,
    example=examples[0],
)

trial.output

Logfire project URL: https://logfire-eu.pydantic.dev/nachollorca/ai-assist-demo

17:03:29.478 chat mistral-small-latest


{'sentiment': 'positive',
 'reason': "The review expresses strong satisfaction with the product's battery life, screen quality, and overall experience."}

Now that our trial ran successfully, we can jump into quality measurements.

First, a simple deterministic check: does the predicted label match the ground truth? No LLM judge needed — a `ProgrammaticMetric` whose scorer follows the `ProgrammaticScorer` protocol is the right tool. We declare the value space with an `Ordinal` scale.

In [6]:
from lmeh.datatypes import Ordinal, ProgrammaticMetric, RawScore

def label_match(output: dict, example: Example) -> RawScore:
    predicted = output["sentiment"]
    expected = example.reference
    if predicted == expected:
        return RawScore(raw=True, reason=f"Predicted '{predicted}' matches reference.")
    return RawScore(
        raw=False,
        reason=f"Predicted '{predicted}', expected '{expected}'.",
    )

label_correctness = ProgrammaticMetric(
    name="label_correctness",
    description="Whether the predicted sentiment label matches the ground-truth label.",
    scale=Ordinal(levels=[False, True]),
    scorer=label_match,
)

Now we score the trial against the metric.

In [7]:
from lmeh.execution import score_metric

scoring = score_metric(trial=trial, metric=label_correctness)
scoring.score

Score(raw=True, normalized=1.0, reason="Predicted 'positive' matches reference.")

The label check is cheap and exact, but it tells us nothing about the *reason* the model produced — and a good sentiment classifier should be able to justify its call. That's a subjective, open-ended judgement with no ground truth, which is exactly where an LLM judge earns its keep.

We define an `LLMJudgeMetric` that grades the quality of the justification. It carries its own `LMConfig` and judge prompt template. We use the built-in `default_llm_judge`, which feeds the judge the rendered prompt, the target's output, the reference, and the metric description.


In [8]:
from lmeh.datatypes import LLMJudgeMetric
from lmeh.judges import default_llm_judge

description = """Rate how well the 'reason' field justifies the predicted sentiment label given the original review. A good reason cites concrete cues from the review and is consistent with the predicted label. Score 'good' if the justification is grounded and coherent, 'poor' otherwise."""

reason_quality = LLMJudgeMetric(
    name="reason_quality",
    description=description,
    scale=Ordinal(levels=["poor", "good"]),
    scorer=default_llm_judge,
    config=LMConfig(
        model="mistral:mistral-medium-latest",
        generation_kwargs={"temperature": 0.1},
    ),
)

Let's see what the judge thinks about the system output on our first trial.


In [9]:
judge_scoring = score_metric(trial=trial, metric=reason_quality)
judge_scoring.score

17:03:30.174 chat mistral-medium-latest


Score(raw='good', normalized=1.0, reason="The 'reason' field explicitly cites concrete cues from the review ('battery life', 'screen quality', and 'overall experience') and aligns them with the predicted 'positive' sentiment label, demonstrating a grounded and coherent justification.")

Finally, we can wire it all together into an experiment: run `classify_sentiment` against every example and score both metrics on each output.


In [10]:
from lmeh.datatypes import Experiment
from lmeh.execution import run_experiment

experiment = Experiment(
    name="sentiment-baseline",
    target=classify_sentiment,
    prompt_template=prompt_template,
    config=config,
)

results = run_experiment(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_quality],
    workers=5,
)

17:03:31.850 chat mistral-small-latest
17:03:31.854 chat mistral-small-latest
17:03:31.858 chat mistral-small-latest
17:03:31.861 chat mistral-small-latest
17:03:31.869 chat mistral-small-latest
17:03:32.294 chat mistral-medium-latest
17:03:32.383 chat mistral-medium-latest
17:03:32.425 chat mistral-medium-latest
17:03:32.429 chat mistral-medium-latest
17:03:32.435 chat mistral-medium-latest


And a reporting utility renders the results.


In [11]:
from lmeh.reporting import markdown_report
from IPython.display import Markdown, display

report = markdown_report(results)
display(Markdown(report))
display(Markdown("---"))

# Run report: `sentiment-baseline`

Summary of one experiment run across the dataset.

## Experiment

- **Model**: `mistral:mistral-small-latest`
- **Target repeats**: 1
- **Run timestamp**: 2026-06-10T15:03:42.520113+00:00

## Quality

Headline score is the mean across per-metric means (each metric weighted equally). Dispersion is reported as `mean ± sd (n)`.

- **Overall**: 0.9000 ± 0.1414 (n=2)

### Per-metric

`Score` aggregates per-example means (dispersion = dataset heterogeneity). `Replicate noise` is the average within-cell SD across replicates (dispersion = measurement instability).

| Metric | Score | Replicate noise |
| --- | --- | --- |
| `label_correctness` | 0.8000 ± 0.4472 (n=5) | 0.0000 ± 0.0000 (n=5) |
| `reason_quality` | 1.0000 ± 0.0000 (n=5) | 0.0000 ± 0.0000 (n=5) |

## Reliability

Trial failures count against the run (the target is under evaluation); scorer failures are excluded from quality aggregates.

- **Trials**: 5 successful / 5 total
- **Trial failure rate**: 0.00%
- **Scorings**: 10 successful / 10 total
- **Scoring failure rate**: 0.00%

## Telemetry

Totals across successful trials only.

- **Total latency**: 2.654 s
- **Total output tokens**: 149
- **Throughput**: 56.1 tok/s

---

The harness can also **search for a better prompt template**. `optimize()` evaluates the experiment's current template as a baseline, then repeatedly asks a *proposer* model to rewrite the template from the full trajectory and re-scores each candidate. The archive keeps every checkpoint; `OptimizationResult.best` is the highest-scoring step (not necessarily the last).

In [12]:
from lmeh.optimization import optimize

optimization = optimize(
    experiment=experiment,
    examples=examples,
    metrics=[label_correctness, reason_quality],
    proposer_config=LMConfig(
        model="mistral:mistral-medium-latest",
        generation_kwargs={"temperature": 0.3},
    ),
    steps=5,
    workers=5,
)

17:03:42.570 chat mistral-small-latest
17:03:42.573 chat mistral-small-latest
17:03:42.577 chat mistral-small-latest
17:03:42.581 chat mistral-small-latest
17:03:42.587 chat mistral-small-latest
17:03:43.007 chat mistral-medium-latest
17:03:43.066 chat mistral-medium-latest
17:03:43.072 chat mistral-medium-latest
17:03:43.106 chat mistral-medium-latest
17:03:43.428 chat mistral-medium-latest
17:03:51.182 chat mistral-medium-latest
17:03:56.519 chat mistral-small-latest
17:03:56.523 chat mistral-small-latest
17:03:56.526 chat mistral-small-latest
17:03:56.528 chat mistral-small-latest
17:03:56.531 chat mistral-small-latest
17:03:57.013 chat mistral-medium-latest
17:03:57.033 chat mistral-medium-latest
17:03:57.043 chat mistral-medium-latest
17:03:57.072 chat mistral-medium-latest
17:03:57.077 chat mistral-medium-latest
17:03:59.925 chat mistral-medium-latest
17:04:04.167 chat mistral-small-latest
17:04:04.171 chat mistral-small-latest
17:04:04.175 chat mistral-small-latest
17:04:04.178 

In [13]:
baseline = optimization.steps[0]
best = optimization.best

print(f"Baseline score: {baseline.score:.4f}")
print(f"Best score:     {best.score:.4f} (step {optimization.steps.index(best)})")
print()
print("Best prompt template:")
print(best.prompt_template)

Baseline score: 0.9000
Best score:     0.9000 (step 0)

Best prompt template:
Your task is to classify the sentiment of this product review:

{{ REVIEW }}

The possible labels are 'positive', 'neutral' and 'negative'

